In [18]:
import numpy as np
import pandas as pd
import ast
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
import random

In [19]:
torch.manual_seed(42); np.random.seed(42); random.seed(42)

In [20]:
unique_slugs_embeddings = pd.read_csv('unique_slugs_embeddings.csv')
product_description_embeddings = pd.read_csv('product_desc_embeddings.csv')
journeys_df = pd.read_csv('user_journeys.csv')
journeys_df['slug'] = journeys_df['slug'].apply(ast.literal_eval)

unique_slugs_embeddings['embeddings'] = unique_slugs_embeddings['embeddings'].str.strip('[]').str.replace('\n',' ').str.split().apply(
    lambda x: np.array(x, dtype=np.float32)
)
product_description_embeddings['embeddings'] = product_description_embeddings['embeddings'].str.strip('[]').str.replace('\n',' ').str.split().apply(
    lambda x: np.array(x, dtype=np.float32)
)
unique_slugs_embeddings.rename(columns={'product_id': 'id'}, inplace=True)
unique_slugs_embeddings.drop(columns=['Unnamed: 0'], inplace=True)

unique_pids = product_description_embeddings['id'].unique().tolist()
pid_to_idx = {pid: i for i, pid in enumerate(unique_pids)}
slug_emb_dict = dict(zip(unique_slugs_embeddings['slug'], unique_slugs_embeddings['embeddings']))
slug_to_pid = unique_slugs_embeddings.set_index('slug')['id'].to_dict()
slug_to_pid['UNKNOWN'] = -1

In [21]:
sum([x > 10000 for x in unique_slugs_embeddings['id'].unique()])

5

In [22]:
mapping = pd.read_csv('old_site_new_site_products.csv')
mapping['old_site_id'] = mapping['old_site_id'].astype(float)
mapping['new_site_id'] = mapping['new_site_id'].astype(float)

In [23]:
mapping[mapping['old_site_id'] == 3673]

,old_site_id,new_site_id


In [24]:
old_to_new = dict(zip(mapping['old_site_id'], mapping['new_site_id']))
old_to_new[-1.0] = -1.0
slug_to_pid = unique_slugs_embeddings.set_index('slug')['id'].apply(lambda x: old_to_new[x] if x in old_to_new.keys() else x).to_dict()

In [25]:
len(slug_to_pid)

728

In [26]:
sum([x > 10000 for x in slug_to_pid.values()])

399

In [27]:
slug_to_idx = {pid: i for i, pid in enumerate(slug_to_pid.keys())}

In [28]:
class LSTMRecommender(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(LSTMRecommender, self).__init__()
        # Maps slug indices to dense vectors. padding_idx=0 ensures 0s don't learn.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim, 
            hidden_size=hidden_dim, 
            num_layers=2, 
            batch_first=True, 
            dropout=0.2
        )
        
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x, lengths):
        # x is (batch, seq_len) with Long (integer) values
        x = self.embedding(x) 
        
        # Pack the sequences for efficiency
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True)
        _, (hn, _) = self.lstm(packed_x)
        
        # Use final hidden state of the last LSTM layer
        return self.fc(hn[-1])

In [29]:
from torch.utils.data import Dataset, DataLoader

class SlidingWindowDataset(Dataset):
    def __init__(self, journeys_df, slug_to_idx, pid_to_idx, max_window=10):
        self.samples = []
        self.slug_to_idx = slug_to_idx
        
        for _, row in journeys_df.iterrows():
            seq = row['slug']
            # print(len(seq))
            for i in range(1, len(seq)):
                input_slugs = seq[max(0, i - max_window):i]
                target_slug = seq[i]
                
                target_pid = slug_to_pid.get(target_slug)
                if target_pid in pid_to_idx:
                    # Convert slugs to indices immediately
                    idx_seq = [self.slug_to_idx.get(s, 0) for s in input_slugs]
                    self.samples.append((idx_seq, pid_to_idx[target_pid]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x_idxs, target_idx = self.samples[idx]
        # CRITICAL: Must be LongTensor
        return torch.tensor(x_idxs, dtype=torch.long), torch.tensor(target_idx, dtype=torch.long)

def collate_fn(batch):
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    sequences, targets = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences])
    
    # padding_value=0 matches our padding_idx in the embedding layer
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=0)
    
    return padded_seqs, torch.tensor(targets, dtype=torch.long), lengths

In [30]:
# 1. Setup Parameters
# vocab_size = max index in slug_to_idx + 1 (to account for 0 padding)
vocab_size = max(slug_to_idx.values()) + 1 
embedding_dim = 256
hidden_dim = 512
num_classes = len(pid_to_idx)

In [31]:
# 2. Initialize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMRecommender(vocab_size, embedding_dim, hidden_dim, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [32]:
# from sklearn.model_selection import train_test_split

# # Split the original journeys (e.g., 80% train, 20% validation)
# train_df, val_df = train_test_split(journeys_df, test_size=0.2, random_state=42)

# print(f"Original journeys: {len(journeys_df)}")
# print(f"Train journeys: {len(train_df)}")
# print(f"Validation journeys: {len(val_df)}")

In [33]:
# Create the training dataset and loader
train_dataset = SlidingWindowDataset(journeys_df=journeys_df, slug_to_idx=slug_to_idx, pid_to_idx=pid_to_idx)
train_loader = DataLoader(train_dataset, batch_size=64, collate_fn=collate_fn, shuffle=True)

# Create the validation dataset and loader
# val_dataset = SlidingWindowDataset(journeys_df=val_df, slug_to_idx=slug_to_idx, pid_to_idx=pid_to_idx)
# val_loader = DataLoader(val_dataset, batch_size=64, collate_fn=collate_fn, shuffle=False)

print(f"Train samples (windows): {len(train_dataset)}")
# print(f"Val samples (windows): {len(val_dataset)}")

Train samples (windows): 44906


In [34]:
def evaluate_recall_at_6(model, dataloader, device):
    model.eval()
    hits = 0
    total = 0
    with torch.no_grad():
        for x, y, lengths in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x, lengths)
            
            # Get the top 6 highest probability indices
            _, top6 = torch.topk(logits, k=6, dim=1)
            
            # Check if actual index y is in the top6
            hits += (top6 == y.unsqueeze(1)).any(dim=1).sum().item()
            total += y.size(0)
    return hits / total

In [37]:
# 3. Training Loop
for epoch in range(4):
    model.train()
    epoch_loss = 0
    for x, y, lengths in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        outputs = model(x, lengths)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    # test_loss = 0
    # for x,y, lengths in val_loader:
    #     x, y = x.to(device), y.to(device)
    #     outs = model(x,lengths)
    #     loss = criterion(outs,y)
    #     test_loss += loss
        
    # recall = evaluate_recall_at_6(model, val_loader, device)
    print(f"Epoch {epoch+1} | Train_Loss: {epoch_loss/len(train_loader):.4f}")

Epoch 1 | Train_Loss: 2.6899
Epoch 2 | Train_Loss: 2.4211
Epoch 3 | Train_Loss: 2.2103
Epoch 4 | Train_Loss: 2.0141


In [41]:
import pickle
import torch

# 1. Save the Slug to Index mapping
with open('slug_to_idx.pkl', 'wb') as f:
    pickle.dump(slug_to_idx, f)

# 2. Save the Index to Product ID mapping
# We create this by reversing your pid_to_idx
pid_idx_to_pid = {idx: pid for pid, idx in pid_to_idx.items()}
with open('pid_idx_to_pid.pkl', 'wb') as f:
    pickle.dump(pid_idx_to_pid, f)

# 3. Save the Model Weights
torch.save(model.state_dict(), 'model(v1).pth')

In [42]:
with open('pid_to_idx.pkl', 'wb') as f:
    pickle.dump(pid_to_idx, f)

In [43]:
with open('pid_to_idx.pkl', 'wb') as f:
    pickle.dump(pid_to_idx, f)